# Understand Normalized Metrics

This notebook explains the first formal metric layer for the project. It uses small real PubMedQA and SciFact rows, converts them into the normalized `v0.1` artifact shape, and evaluates those artifacts.

Important: this notebook does **not** call any LLM API. It uses oracle context/evidence and gold-label demo predictions so you can focus on the schema and metric mechanics.

## 1. Setup

The code below points at the project root, imports the sample builder, and imports the evaluator. The generated mini artifacts are written under `outputs/`, which is ignored by Git.

In [1]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import display

from rag_experiment.data.build_normalized_samples import (
    DEFAULT_HF_CACHE,
    DEFAULT_OUTPUT_DIR,
    DEFAULT_SCIFACT_DIR,
    build_pubmedqa_sample,
    build_scifact_sample,
)
from rag_experiment.evaluation.evaluate_artifact import evaluate_file

PROJECT_ROOT = Path.cwd()
PUBMEDQA_PATH = DEFAULT_OUTPUT_DIR / 'pubmedqa_mini_v01.jsonl'
SCIFACT_PATH = DEFAULT_OUTPUT_DIR / 'scifact_mini_v01.jsonl'

PROJECT_ROOT

PosixPath('/Users/KnowYourself/Desktop/Desktop/Obsidian_Vault/Personal Assistant/Learn_Internship/CS6493_Group_Project/rag_project')

## 2. Build Tiny Real-Data Artifacts

These functions read real PubMedQA and SciFact data, but they do not perform real retrieval yet. They create small normalized artifacts using:

- PubMedQA: the question's own context passages as oracle retrieved passages
- SciFact: the gold rationale sentences as oracle retrieved passages
- Both: gold labels as demo predictions

That means the metrics should be perfect here. This is a learning check, not a report-facing experiment result.

In [2]:
build_pubmedqa_sample(limit=5, output_path=PUBMEDQA_PATH, cache_dir=DEFAULT_HF_CACHE)
build_scifact_sample(limit=5, output_path=SCIFACT_PATH, data_dir=DEFAULT_SCIFACT_DIR)

print(PUBMEDQA_PATH)
print(SCIFACT_PATH)

/opt/homebrew/Caskroom/miniconda/base/envs/LLM/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/Users/KnowYourself/Desktop/Desktop/Obsidian_Vault/Personal Assistant/Learn_Internship/CS6493_Group_Project/rag_project/outputs/normalized_samples/pubmedqa_mini_v01.jsonl
/Users/KnowYourself/Desktop/Desktop/Obsidian_Vault/Personal Assistant/Learn_Internship/CS6493_Group_Project/rag_project/outputs/normalized_samples/scifact_mini_v01.jsonl


## 3. Load the JSONL Artifacts

A normalized artifact is one JSON object per example. The evaluator only supports this `v0.1` shape. It intentionally does not try to understand older HotpotQA smoke-test outputs.

In [3]:
def load_jsonl(path: Path):
    with path.open('r', encoding='utf-8') as handle:
        return [json.loads(line) for line in handle if line.strip()]

pubmedqa_rows = load_jsonl(PUBMEDQA_PATH)
scifact_rows = load_jsonl(SCIFACT_PATH)

summary = pd.DataFrame([
    {
        'dataset': 'pubmedqa',
        'rows': len(pubmedqa_rows),
        'run_name': pubmedqa_rows[0]['run']['name'],
        'retriever': pubmedqa_rows[0]['run']['retriever'],
    },
    {
        'dataset': 'scifact',
        'rows': len(scifact_rows),
        'run_name': scifact_rows[0]['run']['name'],
        'retriever': scifact_rows[0]['run']['retriever'],
    },
])

display(summary)

,dataset,rows,run_name,retriever
0,pubmedqa,5,pubmedqa_mini_oracle_context,oracle_context
1,scifact,5,scifact_mini_oracle_evidence,oracle_evidence


## 4. PubMedQA Example

PubMedQA is mainly useful for answer accuracy. Its gold answer is `yes`, `no`, or `maybe`. It has abstract/context text, but not precise rationale sentence IDs, so retrieval metrics are coarse context-level checks.

In [4]:
def preview_text(text, length=180):
    text = ' '.join(str(text).split())
    return text if len(text) <= length else text[:length] + '...'

pub = pubmedqa_rows[0]

print('Query:', pub['example']['query'])
print('Gold answer:', pub['example']['gold_answer'])
print('Demo prediction:', pub['prediction']['answer'])
print('Gold evidence entries:', pub['example']['gold_evidence'])

pub_passages = pd.DataFrame([
    {
        'rank': p['rank'],
        'pubid': p['metadata']['pubid'],
        'context_idx': p['metadata']['context_idx'],
        'context_label': p['metadata'].get('context_label'),
        'text_preview': preview_text(p['text']),
    }
    for p in pub['retrieved_passages']
])

display(pub_passages)

Query: Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?
Gold answer: yes
Demo prediction: yes
Gold evidence entries: [{'pubid': '21645374', 'context_idx': 0}, {'pubid': '21645374', 'context_idx': 1}]


,rank,pubid,context_idx,context_label,text_preview
0,1,21645374,0,BACKGROUND,Programmed cell death (PCD) is the regulated d...
1,2,21645374,1,RESULTS,The following paper elucidates the role of mit...


## 5. SciFact Example

SciFact is mainly useful for evidence retrieval and grounding. Its input is a scientific claim. The gold answer is a verification label such as `SUPPORT` or `CONTRADICT`, and the gold evidence points to rationale sentence indices inside a corpus document.

In [5]:
sci = scifact_rows[0]

print('Claim:', sci['example']['query'])
print('Gold label:', sci['example']['gold_answer'])
print('Demo prediction:', sci['prediction']['answer'])
print('Gold evidence entries:', sci['example']['gold_evidence'])

sci_passages = pd.DataFrame([
    {
        'rank': p['rank'],
        'doc_id': p['metadata']['doc_id'],
        'sentence_index': p['metadata']['sentence_index'],
        'title': p['metadata'].get('title'),
        'text_preview': preview_text(p['text']),
    }
    for p in sci['retrieved_passages']
])

display(sci_passages)

Claim: 1 in 5 million in UK have abnormal PrP positivity.
Gold label: CONTRADICT
Demo prediction: CONTRADICT
Gold evidence entries: [{'doc_id': '13734012', 'sentence_index': 4}]


,rank,doc_id,sentence_index,title,text_preview
0,1,13734012,4,Prevalent abnormal prion protein in human appe...,"RESULTS Of the 32,441 appendix samples 16 were..."


## 6. Evaluate the Mini Artifacts

The evaluator reads normalized `v0.1` rows and aggregates metrics. Because these are oracle demos, the scores should be `1.0`. Later, when we replace oracle passages with BM25/dense/hybrid retrieval over a pooled corpus, these same metrics will become meaningful experiment evidence.

In [6]:
pub_summary = evaluate_file(PUBMEDQA_PATH, dataset='pubmedqa')
sci_summary = evaluate_file(SCIFACT_PATH, dataset='scifact')

Normalized Artifact Evaluation
schema_version: v0.1
dataset: pubmedqa
artifact: /Users/KnowYourself/Desktop/Desktop/Obsidian_Vault/Personal Assistant/Learn_Internship/CS6493_Group_Project/rag_project/outputs/normalized_samples/pubmedqa_mini_v01.jsonl
run: {'name': 'pubmedqa_mini_oracle_context', 'retriever': 'oracle_context', 'top_k': 2, 'model_profile': 'gold_label_demo'}
rows: 5  errors: 0
metrics:
  answer_accuracy: 1.0
  answer_correct: 5
  answer_total: 5
  context_hit_at_k: 1.0
  context_hit_count: 5
  context_total: 5
  context_recall_at_k: 1.0
  context_skipped: 0
Normalized Artifact Evaluation
schema_version: v0.1
dataset: scifact
artifact: /Users/KnowYourself/Desktop/Desktop/Obsidian_Vault/Personal Assistant/Learn_Internship/CS6493_Group_Project/rag_project/outputs/normalized_samples/scifact_mini_v01.jsonl
run: {'name': 'scifact_mini_oracle_evidence', 'retriever': 'oracle_evidence', 'top_k': 1, 'model_profile': 'gold_label_demo'}
rows: 5  errors: 0
metrics:
  label_accuracy: 

## 7. Put the Metrics Side by Side

The metric names are intentionally dataset-aware. PubMedQA has `context_*` metrics because its evidence is coarse. SciFact has `evidence_*` metrics because it has rationale sentence labels.

In [7]:
def metrics_table(summary):
    return pd.DataFrame([
        {'metric': key, 'value': value}
        for key, value in summary['metrics'].items()
    ])

print('PubMedQA metrics')
display(metrics_table(pub_summary))

print('SciFact metrics')
display(metrics_table(sci_summary))

PubMedQA metrics


,metric,value
0,answer_accuracy,1.0
1,answer_correct,5.0
2,answer_total,5.0
3,context_hit_at_k,1.0
4,context_hit_count,5.0
5,context_total,5.0
6,context_recall_at_k,1.0
7,context_skipped,0.0


SciFact metrics


,metric,value
0,label_accuracy,1.0
1,label_correct,5.0
2,label_total,5.0
3,evidence_doc_hit_at_k,1.0
4,evidence_doc_hit_count,5.0
5,evidence_sentence_hit_at_k,1.0
6,evidence_sentence_hit_count,5.0
7,evidence_all_hit_at_k,1.0
8,evidence_all_hit_count,5.0
9,evidence_recall_at_k,1.0


## 8. What This Proves

This notebook proves that:

- real PubMedQA and SciFact rows can be normalized into the same `v0.1` artifact shape
- the metric evaluator can read those artifacts
- PubMedQA and SciFact should not use identical evidence metrics
- the metric layer is ready for a later pooled-corpus retrieval run

This notebook does **not** prove that BM25, dense retrieval, or hybrid retrieval performs well. The current mini artifacts use oracle context/evidence. The next experiment step is to build pooled-corpus retrieval artifacts where the retriever must find evidence among distractors.